# 🚗 EV Driver Behavior Scoring System
## Notebook 1 — Hybrid Dataset Generation

### Approach
We use the **UCI ECO-driving dataset** as our real base (speed, acceleration, braking events).
On top of that, we engineer **EV-specific features** using physics formulas:
- Regenerative braking ratio
- State of Charge (SoC) pattern
- kWh/100km energy consumption
- Motor efficiency impact

This is the standard academic approach used in EV research when proprietary telemetry data (Tesla, Ola, Tata) is not available.

### EV Concepts Used
| Feature | EV Physics Behind It |
|---|---|
| Regenerative braking ratio | % of braking events that recover energy vs friction brakes |
| Acceleration aggression index | Hard acceleration draws peak current, stresses battery |
| Speed smoothness | Constant speed = efficient; fluctuation = energy waste |
| kWh/100km | Standard EV energy consumption metric (like mileage for ICE) |
| SoC swing | Deep discharge cycles degrade Li-ion batteries faster |

In [ ]:
# ── Cell 1: Install & Import ──────────────────────────────────────────────────
!pip install ucimlrepo --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('✅ Libraries loaded successfully')

## Step 1 — Load Real UCI ECO-Driving Data

The UCI ECO-driving dataset contains real OBD-II recorded driving sessions:
speed (km/h), acceleration (m/s²), throttle position, braking events.

We use this as the **real foundation** of our dataset.

In [ ]:
# ── Cell 2: Load UCI ECO Driving Dataset ─────────────────────────────────────
from ucimlrepo import fetch_ucirepo

# Fetch dataset (ID 554 = ECO driving dataset)
try:
    eco = fetch_ucirepo(id=554)
    df_raw = eco.data.features
    print(f'✅ UCI dataset loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
    print(df_raw.head())
except Exception as e:
    print(f'UCI fetch failed ({e}), generating fallback base data...')
    # Fallback: generate realistic base driving data if UCI fetch fails
    n = 5000
    df_raw = pd.DataFrame({
        'speed_kmh':       np.clip(np.random.normal(45, 20, n), 0, 120),
        'acceleration':    np.random.normal(0.5, 1.2, n),
        'throttle_pos':    np.clip(np.random.normal(30, 15, n), 0, 100),
        'brake_pressure':  np.clip(np.random.exponential(8, n), 0, 80),
    })
    print(f'✅ Fallback data created: {df_raw.shape}')

In [ ]:
# ── Cell 3: Inspect & Clean Base Data ────────────────────────────────────────
print('=== Dataset Info ===')
print(df_raw.info())
print('\n=== Missing Values ===')
print(df_raw.isnull().sum())
print('\n=== Basic Statistics ===')
print(df_raw.describe())

## Step 2 — Simulate Drive Sessions

We aggregate the raw time-series data into **per-session features**.
Each session = one driver completing a trip (like one Ola Electric ride).

We create **1200 sessions** by sampling windows from the raw data.

In [ ]:
# ── Cell 4: Create Per-Session Features from Real Data ───────────────────────
N_SESSIONS = 1200
sessions = []

# Column mapping (handles both UCI and fallback)
speed_col  = [c for c in df_raw.columns if 'speed' in c.lower() or 'vel' in c.lower()]
accel_col  = [c for c in df_raw.columns if 'accel' in c.lower()]
throttle_col = [c for c in df_raw.columns if 'throttle' in c.lower() or 'gas' in c.lower()]
brake_col  = [c for c in df_raw.columns if 'brake' in c.lower()]

speed_col   = speed_col[0]   if speed_col   else None
accel_col   = accel_col[0]   if accel_col   else None
throttle_col = throttle_col[0] if throttle_col else None
brake_col   = brake_col[0]   if brake_col   else None

print(f'Using columns → speed: {speed_col}, accel: {accel_col}, throttle: {throttle_col}, brake: {brake_col}')

for i in range(N_SESSIONS):
    # Sample a random window of 100-300 rows (one trip)
    window = min(np.random.randint(100, 300), len(df_raw) - 1)
    start  = np.random.randint(0, max(1, len(df_raw) - window))
    trip   = df_raw.iloc[start:start + window]

    # Extract real base features
    avg_speed    = float(trip[speed_col].mean())  if speed_col   else np.random.normal(42, 18)
    std_speed    = float(trip[speed_col].std())   if speed_col   else np.random.normal(15, 5)
    avg_accel    = float(trip[accel_col].mean())  if accel_col   else np.random.normal(0.5, 0.8)
    std_accel    = float(trip[accel_col].std())   if accel_col   else np.random.normal(1.0, 0.4)
    avg_throttle = float(trip[throttle_col].mean()) if throttle_col else np.random.normal(32, 12)
    avg_brake    = float(trip[brake_col].mean())  if brake_col   else np.random.exponential(7)

    sessions.append({
        'session_id':  i + 1,
        'avg_speed_kmh':   np.clip(avg_speed,  0, 120),
        'std_speed_kmh':   np.clip(std_speed,  0, 60),
        'avg_accel_ms2':   avg_accel,
        'std_accel_ms2':   np.clip(std_accel,  0, 5),
        'avg_throttle_pct': np.clip(avg_throttle, 0, 100),
        'avg_brake_pressure': np.clip(avg_brake, 0, 80),
        'trip_duration_min': window * 0.5,  # assume 0.5s sampling rate
    })

df_sessions = pd.DataFrame(sessions)
print(f'\n✅ {N_SESSIONS} drive sessions created')
print(df_sessions.head())

## Step 3 — Engineer EV-Specific Features

This is the core EV technical contribution of the project.
We derive EV-specific metrics from the real driving base using physics formulas.

### Physics Used
- **Regen braking ratio:** Gentle deceleration (<0.3 m/s²) can be handled by the motor acting as generator. Harsh braking uses friction pads — energy is lost as heat.
- **Aggression index:** Combines throttle % and std of acceleration. High values → battery peak current draw → efficiency loss.
- **kWh/100km:** Modelled from base consumption (14 kWh/100km for a typical EV like Tata Nexon) adjusted by driving behavior factors.

In [ ]:
# ── Cell 5: Engineer EV-Specific Features ────────────────────────────────────
df = df_sessions.copy()

# 1. Regenerative Braking Ratio (0–1)
# Higher brake pressure = more friction braking = less regen recovery
# Formula: regen_ratio = 1 - (brake_pressure / 80) with noise
df['regen_braking_ratio'] = np.clip(
    1 - (df['avg_brake_pressure'] / 80) + np.random.normal(0, 0.05, N_SESSIONS),
    0.1, 0.95
)

# 2. Acceleration Aggression Index (0–100)
# Combines throttle position and acceleration variability
df['aggression_index'] = np.clip(
    (df['avg_throttle_pct'] * 0.6) + (df['std_accel_ms2'] * 8)
    + np.random.normal(0, 3, N_SESSIONS),
    0, 100
)

# 3. Speed Smoothness Score (0–100, higher = smoother)
# Low std_speed = consistent speed = less energy wasted accelerating/decelerating
df['smoothness_score'] = np.clip(
    100 - (df['std_speed_kmh'] * 1.8) + np.random.normal(0, 3, N_SESSIONS),
    0, 100
)

# 4. High Speed Ratio (fraction of trip spent above 80 km/h)
# EVs lose efficiency sharply above 80 km/h due to aerodynamic drag
df['high_speed_ratio'] = np.clip(
    (df['avg_speed_kmh'] - 40) / 80 + np.random.normal(0, 0.08, N_SESSIONS),
    0, 1
)

# 5. State of Charge (SoC) swing during trip (%)
# Aggressive drivers drain battery more per trip
df['soc_swing_pct'] = np.clip(
    10 + (df['aggression_index'] * 0.25) + (df['high_speed_ratio'] * 15)
    + np.random.normal(0, 3, N_SESSIONS),
    5, 60
)

# 6. kWh/100km — TARGET VARIABLE
# Base: 14 kWh/100km (Tata Nexon EV spec)
# Adjustments based on driving behavior (physics-based)
BASE_CONSUMPTION = 14.0
df['kwh_per_100km'] = np.clip(
    BASE_CONSUMPTION
    + (df['aggression_index'] * 0.08)       # aggressive driving increases consumption
    - (df['regen_braking_ratio'] * 3.5)     # regen reduces net consumption
    + (df['high_speed_ratio'] * 4.0)        # high speed increases aero drag losses
    - (df['smoothness_score'] * 0.03)       # smooth driving reduces consumption
    + np.random.normal(0, 0.8, N_SESSIONS), # real-world noise
    8, 28
)

# 7. Trip distance (km)
df['trip_distance_km'] = np.clip(
    df['avg_speed_kmh'] * (df['trip_duration_min'] / 60)
    + np.random.normal(0, 2, N_SESSIONS),
    1, 100
)

print('✅ EV features engineered successfully')
print(df[['session_id','regen_braking_ratio','aggression_index',
          'smoothness_score','soc_swing_pct','kwh_per_100km']].head(8))

## Step 4 — Compute EV Efficiency Score (0–100)

We compute a **weighted composite score** that represents how efficiently a driver uses the EV.

**Score = 100 means perfect EV driving (maximum range, minimum battery stress)**

| Component | Weight | EV Rationale |
|---|---|---|
| Regen braking ratio | 30% | Most impactful for real-world range recovery |
| Smoothness score | 25% | Steady speed = motor runs at peak efficiency |
| Aggression index (inverted) | 25% | Less aggression = less peak current draw |
| High speed ratio (inverted) | 20% | Avoiding high speed reduces aero drag losses |

In [ ]:
# ── Cell 6: Compute EV Efficiency Score ──────────────────────────────────────

# Normalize each component to 0-100 scale
regen_score      = df['regen_braking_ratio'] * 100          # already 0-1
smooth_score     = df['smoothness_score']                    # already 0-100
aggr_score       = 100 - df['aggression_index']              # inverted
speed_score      = (1 - df['high_speed_ratio']) * 100        # inverted

# Weighted sum
df['ev_efficiency_score'] = np.clip(
    (regen_score  * 0.30) +
    (smooth_score * 0.25) +
    (aggr_score   * 0.25) +
    (speed_score  * 0.20),
    0, 100
).round(2)

# Driver Grade Labels
def grade(score):
    if score >= 80: return 'A - Eco Master'
    elif score >= 65: return 'B - Smooth Commuter'
    elif score >= 50: return 'C - Average Driver'
    elif score >= 35: return 'D - Aggressive Urban'
    else: return 'F - Energy Waster'

df['driver_grade'] = df['ev_efficiency_score'].apply(grade)

print('✅ EV Efficiency Score computed')
print('\nScore Distribution:')
print(df['driver_grade'].value_counts())
print(f'\nMean Score: {df["ev_efficiency_score"].mean():.1f}')
print(f'Score Range: {df["ev_efficiency_score"].min():.1f} – {df["ev_efficiency_score"].max():.1f}')

In [ ]:
# ── Cell 7: Quick Sanity Check Plot ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('EV Driver Behavior Dataset — Feature Distributions', fontsize=14, fontweight='bold')

features = [
    ('ev_efficiency_score', 'EV Efficiency Score (0-100)', '#2196F3'),
    ('kwh_per_100km',       'Energy Consumption (kWh/100km)', '#FF5722'),
    ('regen_braking_ratio', 'Regen Braking Ratio', '#4CAF50'),
    ('aggression_index',    'Aggression Index', '#F44336'),
    ('smoothness_score',    'Speed Smoothness Score', '#9C27B0'),
    ('soc_swing_pct',       'SoC Swing Per Trip (%)', '#FF9800'),
]

for ax, (col, label, color) in zip(axes.flatten(), features):
    ax.hist(df[col], bins=35, color=color, alpha=0.75, edgecolor='white')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.2, label=f'Mean: {df[col].mean():.1f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Distribution plots saved')

In [ ]:
# ── Cell 8: Export Final Dataset ─────────────────────────────────────────────

# Final column selection
final_cols = [
    'session_id', 'trip_duration_min', 'trip_distance_km',
    'avg_speed_kmh', 'std_speed_kmh',
    'avg_accel_ms2', 'std_accel_ms2',
    'avg_throttle_pct', 'avg_brake_pressure',
    'regen_braking_ratio', 'aggression_index',
    'smoothness_score', 'high_speed_ratio', 'soc_swing_pct',
    'kwh_per_100km', 'ev_efficiency_score', 'driver_grade'
]

df_final = df[final_cols].copy()
df_final.to_csv('ev_driver_dataset.csv', index=False)

print('✅ Dataset exported as ev_driver_dataset.csv')
print(f'   Shape: {df_final.shape}')
print('\nFinal Dataset Preview:')
print(df_final.head(5).to_string())
print('\nColumn Summary:')
for col in final_cols:
    print(f'  {col:<30} → dtype: {df_final[col].dtype}')

## ✅ Notebook 1 Complete!

### What we built:
- Loaded real UCI ECO-driving data as the base
- Engineered 6 EV-specific features using physics formulas
- Computed a weighted EV Efficiency Score (0–100)
- Created 1200 drive sessions → saved as `ev_driver_dataset.csv`

### Next → Notebook 2: EDA & Visualization
Load `ev_driver_dataset.csv` and explore patterns, correlations, and driver behavior insights.